# E01 · 道路图像处理 V2.0

## 00 · 环境与素材
同一张真实道路图片只修改一个参数，比较处理前后差异。

In [ ]:

from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
def locate_images():
    here=Path.cwd().resolve()
    for root in [here,*here.parents]:
        p=root/'images'
        if (p/'01_solidWhiteRight_straight_white.jpg').exists(): return p
    kaggle=Path('/kaggle/input')
    if kaggle.exists():
        for p in kaggle.glob('**/images/01_solidWhiteRight_straight_white.jpg'): return p.parent
    raise FileNotFoundError('请解压完整教学包，确保 images/ 与 Notebook 一同提供')
IMAGES=locate_images()
def read(name):
    x=cv2.imread(str(IMAGES/name))
    if x is None: raise FileNotFoundError(name)
    return x
def view(items):
    fig,axs=plt.subplots(1,len(items),figsize=(5*len(items),4))
    if len(items)==1: axs=[axs]
    for ax,(name,img) in zip(axs,items):
        ax.imshow(img if img.ndim==2 else cv2.cvtColor(img,cv2.COLOR_BGR2RGB),cmap='gray' if img.ndim==2 else None)
        ax.set_title(name);ax.axis('off')
    plt.tight_layout();plt.show()
print('Image path:',IMAGES,'OpenCV:',cv2.__version__)


## 01 · RGB / HSV 颜色空间
探索黄线和白线为什么需要不同的阈值。

In [ ]:
img=read('02_solidYellowLeft_yellow_white.jpg')
hsv=cv2.cvtColor(img,cv2.COLOR_BGR2HSV)
yellow=cv2.inRange(hsv,(15,70,70),(40,255,255))
white=cv2.inRange(hsv,(0,0,175),(179,85,255))
view([('Original',img),('Yellow',yellow),('White',white),('Combined',cv2.bitwise_or(yellow,white))])

## 02 · 全局 / Otsu / 自适应阈值
同一阈值为什么无法适应所有亮度？

In [ ]:
img=read('06_challenge_bright_pavement.jpg')
g=cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)
_,a=cv2.threshold(g,150,255,cv2.THRESH_BINARY)
_,b=cv2.threshold(g,0,255,cv2.THRESH_BINARY+cv2.THRESH_OTSU)
c=cv2.adaptiveThreshold(g,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY,31,7)
view([('Original',img),('Global',a),('Otsu',b),('Adaptive',c)])

## 03 · 图像滤波与 CLAHE
对比高斯、中值和双边滤波。

In [ ]:
img=read('08_harder_shadow_glare.jpg')
g=cv2.GaussianBlur(img,(5,5),1.2);m=cv2.medianBlur(img,5)
b=cv2.bilateralFilter(img,9,60,60)
c=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8)).apply(cv2.cvtColor(img,cv2.COLOR_BGR2GRAY))
view([('Original',img),('Gaussian',g),('Median',m),('Bilateral',b),('CLAHE',c)])

## 04 · 形态学
腐蚀、膨胀、开运算、闭运算有什么不同？

In [ ]:
img=read('02_solidYellowLeft_yellow_white.jpg')
mask=cv2.inRange(cv2.cvtColor(img,cv2.COLOR_BGR2HSV),(15,70,70),(40,255,255))
k=cv2.getStructuringElement(cv2.MORPH_RECT,(5,5))
view([('Mask',mask),('Erosion',cv2.erode(mask,k)),('Dilation',cv2.dilate(mask,k)),
('Opening',cv2.morphologyEx(mask,cv2.MORPH_OPEN,k)),('Closing',cv2.morphologyEx(mask,cv2.MORPH_CLOSE,k))])

## 05 · 透视变换
没有相机标定时只能得到近似鸟瞰图，不能报告米制距离。

In [ ]:
img=read('01_solidWhiteRight_straight_white.jpg')
h,w=img.shape[:2]
src=np.float32([[.39*w,.56*h],[.61*w,.56*h],[.96*w,.97*h],[.04*w,.97*h]])
dst=np.float32([[.25*w,0],[.75*w,0],[.75*w,h-1],[.25*w,h-1]])
bird=cv2.warpPerspective(img,cv2.getPerspectiveTransform(src,dst),(w,h))
view([('Original',img),('Approximate bird-eye',bird)])